# Cehuamilli

## Configuración del entorno

In [4]:
import ee
import geemap
ee.Authenticate()

True

In [6]:
ee.Initialize(project='climaton2026') # Seleccionamos nuestro proyecto

In [14]:
import geopandas as gpd
import matplotlib.pyplot as plt

# 1. Cargar las 3 capas que tienes en tu carpeta
loc = gpd.read_file("../data/localidad_amanzanada/2025_09013_L01092026_1624.shp")
ageb_urb = gpd.read_file("../data/ageb_urbano/2025_09013_A01092026_1624.shp")
ageb_rur = gpd.read_file("../data/ageb_rural/2025_09013_AR01092026_1624.shp")

print(loc)

# 2. Ver columnas y contenido de localidades
print("--- Columnas en Localidades ---")
print(loc.columns)

print("\n--- Localidades encontradas en Xochimilco ---")
print(loc[['CVEGEO', 'NOMGEO', 'AMBITO']].to_string())

# 3. Filtrar Tulyehualco
tulyehualco = loc[loc['NOMGEO'].str.contains('Tulyehualco', case=False, na=False)]
print("\n--- Coincidencia para Tulyehualco ---")
print(tulyehualco[['CVEGEO', 'NOMGEO', 'geometry']])

   EDICION VERSION TIPO_GEO     CVEGEO CVE_ENT CVE_MUN CVE_LOC  \
0     2025       1        l  090130202      09     013    0202   
1     2025       1        l  090130146      09     013    0146   
2     2025       1        l  090130231      09     013    0231   
3     2025       1        l  090130001      09     013    0001   
4     2025       1        l  090130234      09     013    0234   
5     2025       1        l  090130160      09     013    0160   
6     2025       1        l  090130137      09     013    0137   
7     2025       1        l  090130193      09     013    0193   
8     2025       1        l  090130157      09     013    0157   
9     2025       1        l  090130184      09     013    0184   
10    2025       1        l  090130273      09     013    0273   
11    2025       1        l  090130136      09     013    0136   
12    2025       1        l  090130162      09     013    0162   
13    2025       1        l  090130143      09     013    0143   
14    2025

In [51]:
import geopandas as gpd
import pandas as pd

ageb_urb = gpd.read_file('../data/ageb_urbano/2025_09013_A01092026_1624.shp').to_crs(epsg=4326)
ageb_rur = gpd.read_file('../data/ageb_rural/2025_09013_AR01092026_1624.shp').to_crs(epsg=4326)

# 2. Ventana geográfica de Tulyehualco + faldas del Teuhtli
bbox_geom = box(-99.030, 19.230, -98.990, 19.270)
bbox_gdf = gpd.GeoDataFrame(geometry=[bbox_geom], crs="EPSG:4326")

# 3. Filtrar urbano y RECORTAR rural con el bounding box
tulye_urb = ageb_urb.cx[-99.030:-98.990, 19.230:19.270]
tulye_rur = gpd.clip(ageb_rur, bbox_gdf)


print(f'AGEBs urbanas encontradas: {len(tulye_urb)}')
print(f'AGEBs rurales encontradas: {len(tulye_rur)}')

# 3. Concatenar ambos GeoDataFrames
tulye = pd.concat([tulye_agebs, tulye_agebr], ignore_index=True)

# 4. Unificar todas las geometrías en un solo polígono continuo
# as_index=False conserva una estructura de GeoDataFrame limpia
tulyehualco_poly = tulye.dissolve(as_index=False)

# Si quedan pequeñas divisiones o multipolígonos y solo quieres la envolvente sólida:
# tulyehualco_poly['geometry'] = tulyehualco_poly.geometry.buffer(0)

# 5. Exportar a GeoJSON
tulyehualco_poly.to_file('tulyehualco_completo.geojson', driver='GeoJSON')
print('Exportado a tulyehualco_completo.geojson con éxito.')

AGEBs urbanas encontradas: 14
AGEBs rurales encontradas: 1
Exportado a tulyehualco_completo.geojson con éxito.


In [52]:
import geopandas as gpd
import pandas as pd
import folium
from shapely.geometry import box
from IPython.display import HTML

# 1. Cargar capas y asegurar WGS84
ageb_urb = gpd.read_file('../data/ageb_urbano/2025_09013_A01092026_1624.shp').to_crs(epsg=4326)
ageb_rur = gpd.read_file('../data/ageb_rural/2025_09013_AR01092026_1624.shp').to_crs(epsg=4326)

# 2. Ventana geográfica de Tulyehualco + faldas del Teuhtli
bbox_geom = box(-100, 18, -98.00, 20)
bbox_gdf = gpd.GeoDataFrame(geometry=[bbox_geom], crs="EPSG:4326")

# 3. Filtrar urbano y RECORTAR rural con el bounding box
tulye_urb = ageb_urb.cx[:,:]
tulye_rur = gpd.clip(ageb_rur, bbox_gdf)

# 4. Generar las 3 variantes disueltas
poly_urbano = tulye_urb.dissolve(as_index=False)
poly_rural = tulye_rur.dissolve(as_index=False)

# Combinado: concatenamos y disolvemos en un solo contorno exterior
tulye_concat = pd.concat([poly_urbano, poly_rural], ignore_index=True)
poly_combinado = tulye_concat.dissolve(as_index=False)

# 5. Exportar a GeoJSON
poly_urbano.to_file('tulyehualco_urbano.geojson', driver='GeoJSON')
poly_rural.to_file('tulyehualco_rural.geojson', driver='GeoJSON')
poly_combinado.to_file('tulyehualco_combinado.geojson', driver='GeoJSON')

# 6. Configurar mapa interactivo
m = folium.Map(location=[19.252, -99.010], zoom_start=14, tiles='CartoDB positron')

# Satélite ESRI
folium.TileLayer(
    tiles='https://server.arcgisonline.com/ArcGIS/rest/services/World_Imagery/MapServer/tile/{z}/{y}/{x}',
    attr='ESRI Satellite',
    name='Satélite ESRI'
).add_to(m)

# Capa 1: Solo Urbano (Rojo)
folium.GeoJson(
    poly_urbano,
    name='1. Solo Urbano',
    style_function=lambda x: {'fillColor': '#e41a1c', 'color': '#b10026', 'weight': 2, 'fillOpacity': 0.45}
).add_to(m)

# Capa 2: Solo Rural (Verde - acotado al Teuhtli)
folium.GeoJson(
    poly_rural,
    name='2. Solo Rural',
    style_function=lambda x: {'fillColor': '#4daf4a', 'color': '#006837', 'weight': 2, 'fillOpacity': 0.45}
).add_to(m)

# Capa 3: Combinado (Azul - apagado por default)
folium.GeoJson(
    poly_combinado,
    name='3. Combinado (Total)',
    show=False,
    style_function=lambda x: {'fillColor': '#377eb8', 'color': '#08519c', 'weight': 2.5, 'fillOpacity': 0.35}
).add_to(m)

# Control de capas (esquina superior derecha)
folium.LayerControl(collapsed=False).add_to(m)

HTML(m._repr_html_())

In [21]:
import geopandas as gpd

gdf = gpd.read_file('tulyehualco_urbano.geojson')
m = gdf.explore(name='Tulyehualco AGEBs', style_kwds={'fillColor': '#e31a1c', 'color': 'red'})
m.save('ver_tulyehualco.html')
print('Abre ver_tulyehualco.html en tu navegador para validar el perímetro.')

Abre ver_tulyehualco.html en tu navegador para validar el perímetro.


In [44]:
import geopandas as gpd
from IPython.display import HTML

# 1. Cargar archivo local y asegurar WGS84
cdmx = gpd.read_file("../data/mapa_colonias.json").to_crs(epsg=4326)

# 2. Diccionario de configuración por alcaldía
# Si la lista está vacía ([] o None), selecciona TODOS los pueblos/colonias de esa alcaldía.
# Si agregas strings, filtra por coincidencia parcial (case-insensitive).
TARGETS = {
    "xochimilco": [],  # [] -> trae todos los pueblos/colonias de Xochimilco
    "tláhuac": [],  # Trae los que coincidan con Tulyehualco en Tláhuac
    # "milpa alta": [], # Ejemplo para agregar otra alcaldía
}

# 3. Filtrar de forma dinámica según el diccionario
condiciones = []
for alc_pattern, pueblos in TARGETS.items():
    cond_alc = cdmx["alc"].str.contains(alc_pattern, case=False, na=False)

    if pueblos:
        # Crea una regex tipo: 'pueblo1|pueblo2'
        regex_pueblos = "|".join(pueblos)
        cond_pueblo = cdmx["colonia"].str.contains(
            regex_pueblos, case=False, na=False
        )
        condiciones.append(cond_alc & cond_pueblo)
    else:
        # Trae la alcaldía completa
        condiciones.append(cond_alc)

# Unión de todas las condiciones con OR (|)
filtro_final = condiciones[0]
for cond in condiciones[1:]:
    filtro_final = filtro_final | cond

gdf_filtrado = cdmx[filtro_final].copy()

# 4. Imprimir listado en texto para inspeccionar qué polígonos encontró
print(f"Total de polígonos seleccionados: {len(gdf_filtrado)}\n")
for alc in TARGETS.keys():
    sub = gdf_filtrado[gdf_filtrado["alc"].str.contains(alc, case=False, na=False)]
    print(f"--- {alc.upper()} ({len(sub)} encontrados) ---")
    print(sorted(sub["colonia"].dropna().unique()))
    print()

# 5. Visualizar en mapa interactivo
# Pinta cada alcaldía de un color y muestra el nombre al pasar el cursor
m = gdf_filtrado.explore(
    column="alc",
    tooltip=["colonia", "alc"],
    popup=["colonia", "alc"],
    style_kwds={"fillOpacity": 0.45, "weight": 1.5},
)

HTML(m._repr_html_())

Total de polígonos seleccionados: 196

--- XOCHIMILCO (115 encontrados) ---
['Ampliacion Nativitas', 'Ampliacion San Marcos Norte', 'Ampliacion Tepepan', 'Año De Juarez', 'Barrio 18', 'Barrio 3 De Mayo', 'Barrio Ahualapa', 'Barrio Apatlaco', 'Barrio Belen', 'Barrio Calpulco', 'Barrio Caltongo', 'Barrio Calyequita', 'Barrio Chapultepec', 'Barrio Del Puente', 'Barrio El Calvario', 'Barrio El Rosario', 'Barrio La Asuncion', 'Barrio La Candelaria', 'Barrio La Concepcion Tlacoapa', 'Barrio La Gallera', 'Barrio La Guadalupana', 'Barrio La Guadalupita', 'Barrio La Planta', 'Barrio La Santisima', 'Barrio Las Cruces', 'Barrio Las Flores', 'Barrio Los Reyes', 'Barrio Niños Heroes', 'Barrio Pocitos', 'Barrio San Andres', 'Barrio San Antonio', 'Barrio San Cristobal', 'Barrio San Diego', 'Barrio San Esteban', 'Barrio San Jose Pueblo San Luis Tlaxialtemalco', 'Barrio San Juan', 'Barrio San Juan Minas', 'Barrio San Juan Moyotepec', 'Barrio San Juan Pueblo San Luis Tlaxialtemalco', 'Barrio San Lorenzo

In [41]:
import geopandas as gpd
from IPython.display import HTML

# Cargar localidades de Xochimilco
loc = gpd.read_file('../data/localidad_amanzanada/2025_09013_L01092026_1624.shp').to_crs(epsg=4326)

# Ver tabla completa con nombres y ámbito (urbano/rural)
print(loc[['CVEGEO', 'NOMGEO', 'AMBITO']].to_string())

# Pintar todas en el mapa coloreadas por nombre para ver qué polígono es cuál
m_inegi = loc.explore(
    column='NOMGEO',
    legend=True,
    tooltip=['NOMGEO', 'CVEGEO', 'AMBITO'],
    popup=True
)

HTML(m_inegi._repr_html_())

       CVEGEO                                     NOMGEO  AMBITO
0   090130202                              Paraje Zacapa   Rural
1   090130146                         Tlatzala (Cuayuca)   Rural
2   090130231                                    Tecanco   Rural
3   090130001                                 Xochimilco  Urbana
4   090130234                           Barrio Nochicala   Rural
5   090130160                     Tochuca (San Salvador)   Rural
6   090130137  Tlachiultepec de Ahuayucan (Las Malvinas)   Rural
7   090130193             El Potrero (Bosques de la Paz)   Rural
8   090130157                                   Tipipili   Rural
9   090130184                                     Atotle   Rural
10  090130273                              Zacazontlipac   Rural
11  090130136                                Tepexomulco   Rural
12  090130162                                 Pepelaxtla   Rural
13  090130143                                 Tlalatlaco   Rural
14  090130181            

In [46]:
import geopandas as gpd
from IPython.display import HTML

# 1. Cargar archivo local y asegurar WGS84
cdmx = gpd.read_file("../data/mapa_colonias.json").to_crs(epsg=4326)

# 2. Diccionario de configuración por alcaldía
# Si la lista está vacía ([] o None), selecciona TODOS los pueblos/colonias de esa alcaldía.
# Si agregas strings, filtra por coincidencia parcial (case-insensitive).
TARGETS = {
    "xochimilco": [],  # [] -> trae todos los pueblos/colonias de Xochimilco
    "tláhuac": [],  # Trae los que coincidan con Tulyehualco en Tláhuac
    "milpa alta": [], # Ejemplo para agregar otra alcaldía
}

# 3. Filtrar de forma dinámica según el diccionario
condiciones = []
for alc_pattern, pueblos in TARGETS.items():
    cond_alc = cdmx["alc"].str.contains(alc_pattern, case=False, na=False)

    if pueblos:
        # Crea una regex tipo: 'pueblo1|pueblo2'
        regex_pueblos = "|".join(pueblos)
        cond_pueblo = cdmx["colonia"].str.contains(
            regex_pueblos, case=False, na=False
        )
        condiciones.append(cond_alc & cond_pueblo)
    else:
        # Trae la alcaldía completa
        condiciones.append(cond_alc)

# Unión de todas las condiciones con OR (|)
filtro_final = condiciones[0]
for cond in condiciones[1:]:
    filtro_final = filtro_final | cond

gdf_filtrado = cdmx[filtro_final].copy()

# 4. Imprimir listado en texto para inspeccionar qué polígonos encontró
print(f"Total de polígonos seleccionados: {len(gdf_filtrado)}\n")
for alc in TARGETS.keys():
    sub = gdf_filtrado[gdf_filtrado["alc"].str.contains(alc, case=False, na=False)]
    print(f"--- {alc.upper()} ({len(sub)} encontrados) ---")
    print(sorted(sub["colonia"].dropna().unique()))
    print()

# 5. Visualizar en mapa interactivo
# Pinta cada alcaldía de un color y muestra el nombre al pasar el cursor
m = gdf_filtrado.explore(
    column="alc",
    tooltip=["colonia", "alc"],
    popup=["colonia", "alc"],
    style_kwds={"fillOpacity": 0.45, "weight": 1.5},
)

HTML(m._repr_html_())

Total de polígonos seleccionados: 232

--- XOCHIMILCO (115 encontrados) ---
['Ampliacion Nativitas', 'Ampliacion San Marcos Norte', 'Ampliacion Tepepan', 'Año De Juarez', 'Barrio 18', 'Barrio 3 De Mayo', 'Barrio Ahualapa', 'Barrio Apatlaco', 'Barrio Belen', 'Barrio Calpulco', 'Barrio Caltongo', 'Barrio Calyequita', 'Barrio Chapultepec', 'Barrio Del Puente', 'Barrio El Calvario', 'Barrio El Rosario', 'Barrio La Asuncion', 'Barrio La Candelaria', 'Barrio La Concepcion Tlacoapa', 'Barrio La Gallera', 'Barrio La Guadalupana', 'Barrio La Guadalupita', 'Barrio La Planta', 'Barrio La Santisima', 'Barrio Las Cruces', 'Barrio Las Flores', 'Barrio Los Reyes', 'Barrio Niños Heroes', 'Barrio Pocitos', 'Barrio San Andres', 'Barrio San Antonio', 'Barrio San Cristobal', 'Barrio San Diego', 'Barrio San Esteban', 'Barrio San Jose Pueblo San Luis Tlaxialtemalco', 'Barrio San Juan', 'Barrio San Juan Minas', 'Barrio San Juan Moyotepec', 'Barrio San Juan Pueblo San Luis Tlaxialtemalco', 'Barrio San Lorenzo

In [49]:
import folium
import geopandas as gpd
from IPython.display import HTML

# 1. Cargar las tres capas locales y proyectar a EPSG:4326
loc = gpd.read_file(
    "../data/localidad_amanzanada/2025_09013_L01092026_1624.shp"
).to_crs(epsg=4326)
ageb_urb = gpd.read_file(
    "../data/ageb_urbano/2025_09013_A01092026_1624.shp"
).to_crs(epsg=4326)
ageb_rur = gpd.read_file(
    "../data/ageb_rural/2025_09013_AR01092026_1624.shp"
).to_crs(epsg=4326)

# 2. Inicializar mapa base centrado en Xochimilco
m = folium.Map(
    location=[19.255, -99.05], zoom_start=12, tiles="CartoDB positron"
)

# Añadir capa satelital
folium.TileLayer(
    tiles="https://server.arcgisonline.com/ArcGIS/rest/services/World_Imagery/MapServer/tile/{z}/{y}/{x}",
    attr="ESRI Satellite",
    name="Satélite ESRI",
).add_to(m)

# 3. Capa 1: Localidad Amanzanada (Azul)
# Solo mostramos columnas relevantes para que el popup no sea un muro de texto
cols_loc = [c for c in ["CVEGEO", "NOMGEO", "AMBITO"] if c in loc.columns]
folium.GeoJson(
    loc,
    name="1. Localidades Amanzanadas (Azul)",
    style_function=lambda x: {
        "fillColor": "#1f78b4",
        "color": "#08519c",
        "weight": 2,
        "fillOpacity": 0.4,
    },
    tooltip=folium.GeoJsonTooltip(fields=cols_loc, aliases=cols_loc),
    popup=folium.GeoJsonPopup(fields=cols_loc, aliases=cols_loc),
).add_to(m)

# 4. Capa 2: AGEB Urbano (Naranja/Rojo - encendido por default)
cols_urb = [c for c in ["CVEGEO", "CVE_AGEB"] if c in ageb_urb.columns]
folium.GeoJson(
    ageb_urb,
    name="2. AGEBs Urbanas (Rojo)",
    style_function=lambda x: {
        "fillColor": "#e31a1c",
        "color": "#b10026",
        "weight": 1.2,
        "fillOpacity": 0.35,
    },
    tooltip=folium.GeoJsonTooltip(fields=cols_urb, aliases=cols_urb),
    popup=folium.GeoJsonPopup(fields=cols_urb, aliases=cols_urb),
).add_to(m)

# 5. Capa 3: AGEB Rural (Verde - apagado por default para no tapar todo de golpe)
cols_rur = [c for c in ["CVEGEO", "CVE_AGEB"] if c in ageb_rur.columns]
folium.GeoJson(
    ageb_rur,
    name="3. AGEBs Rurales (Verde)",
    show=False,  # Apagada al abrir, la prendes en el panel
    style_function=lambda x: {
        "fillColor": "#33a02c",
        "color": "#006837",
        "weight": 2,
        "fillOpacity": 0.25,
    },
    tooltip=folium.GeoJsonTooltip(fields=cols_rur, aliases=cols_rur),
    popup=folium.GeoJsonPopup(fields=cols_rur, aliases=cols_rur),
).add_to(m)

# 6. Control de capas y renderizado en VS Code
folium.LayerControl(collapsed=False).add_to(m)
HTML(m._repr_html_())

  EDICION VERSION TIPO_GEO     CVEGEO CVE_ENT CVE_MUN CVE_LOC  \
0    2025       1        l  090130202      09     013    0202   
1    2025       1        l  090130146      09     013    0146   
2    2025       1        l  090130231      09     013    0231   
3    2025       1        l  090130001      09     013    0001   
4    2025       1        l  090130234      09     013    0234   

               NOMGEO  AMBITO  \
0       Paraje Zacapa   Rural   
1  Tlatzala (Cuayuca)   Rural   
2             Tecanco   Rural   
3          Xochimilco  Urbana   
4    Barrio Nochicala   Rural   

                                            geometry  
0  POLYGON ((-99.0831 19.22117, -99.08289 19.2212...  
1  POLYGON ((-99.07726 19.23782, -99.07711 19.237...  
2  POLYGON ((-99.07241 19.23386, -99.0725 19.2329...  
3  MULTIPOLYGON (((-99.10959 19.20375, -99.10895 ...  
4  POLYGON ((-99.04018 19.24507, -99.03991 19.245...  
Index(['EDICION', 'VERSION', 'TIPO_GEO', 'CVEGEO', 'CVE_ENT', 'CVE_MUN',
       

In [50]:

print(loc.head())
print(ageb_urb.head())
print(ageb_rur.head())

  EDICION VERSION TIPO_GEO     CVEGEO CVE_ENT CVE_MUN CVE_LOC  \
0    2025       1        l  090130202      09     013    0202   
1    2025       1        l  090130146      09     013    0146   
2    2025       1        l  090130231      09     013    0231   
3    2025       1        l  090130001      09     013    0001   
4    2025       1        l  090130234      09     013    0234   

               NOMGEO  AMBITO  \
0       Paraje Zacapa   Rural   
1  Tlatzala (Cuayuca)   Rural   
2             Tecanco   Rural   
3          Xochimilco  Urbana   
4    Barrio Nochicala   Rural   

                                            geometry  
0  POLYGON ((-99.0831 19.22117, -99.08289 19.2212...  
1  POLYGON ((-99.07726 19.23782, -99.07711 19.237...  
2  POLYGON ((-99.07241 19.23386, -99.0725 19.2329...  
3  MULTIPOLYGON (((-99.10959 19.20375, -99.10895 ...  
4  POLYGON ((-99.04018 19.24507, -99.03991 19.245...  
  EDICION VERSION TIPO_GEO         CVEGEO CVE_ENT CVE_MUN CVE_LOC CVE_AGEB  \
0 

In [53]:
import geopandas as gpd
from IPython.display import HTML

# 1. Cargar la capa oficial de colonias CDMX
cdmx = gpd.read_file("../data/mapa_colonias.json").to_crs(epsg=4326)

# 2. Diccionario con la lista exacta de colonias y barrios
# Usamos expresiones regulares flexibles para capturar variantes con o sin acento
COLONIAS_TULYEHUALCO = [
    r"Calyequita",
    r"Nativitas",
    r"La Esperanza",
    r"La Loma",
    r"Quirino Mendoza",
    r"Los Cerrillos|Cerrillos",
    r"Cristo Rey",
    r"San Felipe de Jesús|San Felipe",
    r"Chiquimola",
    r"Casahuates",
    r"Olivar Santa Mar[ií]a",
    r"Las [AÁ]nimas",
    r"San Sebasti[aá]n",
    r"San Isidro",
    r"La Lupita",
    r"Las Mesitas",
    r"Santiaguito",
    r"Del Carmen",
]

TARGETS = {
    # Todas las colonias/barrios en Xochimilco
    "xochimilco": COLONIAS_TULYEHUALCO,
    # El polígono censal del pueblo que la CDMX colocó bajo Tláhuac
    "tláhuac": [r"TULYEHUALCO"],
}

# 3. Filtrar dinámicamente según el diccionario
condiciones = []
for alc_pattern, patrones in TARGETS.items():
    cond_alc = cdmx["alc"].str.contains(alc_pattern, case=False, na=False)
    regex_col = "|".join(patrones)
    cond_col = cdmx["colonia"].str.contains(regex_col, case=False, na=False)
    condiciones.append(cond_alc & cond_col)

filtro_final = condiciones[0]
for cond in condiciones[1:]:
    filtro_final = filtro_final | cond

tulye_partes = cdmx[filtro_final].copy()

# 4. Imprimir qué polígonos encontró para validar que no falte ninguno
print(f"Total de unidades encontradas: {len(tulye_partes)}")
print(tulye_partes[["alc", "colonia"]].to_string())

# 5. Generar el contorno unificado (AOI listo para GEE)
tulye_aoi = tulye_partes.dissolve(as_index=False)
tulye_aoi.to_file("tulyehualco_aoi_oficial.geojson", driver="GeoJSON")
print("\nExportado a 'tulyehualco_aoi_oficial.geojson' con éxito.")

# 6. Visualizar las piezas encontradas y el contorno final
m = tulye_partes.explore(
    column="colonia",
    tooltip=["colonia", "alc"],
    legend=False,
    style_kwds={"fillOpacity": 0.5, "weight": 1.5},
)

HTML(m._repr_html_())

Total de unidades encontradas: 18
             alc                       colonia
1015     Tláhuac            Pueblo Tulyehualco
1189  Xochimilco          Ampliacion Nativitas
1190  Xochimilco          Ampliacion Nativitas
1201  Xochimilco             Barrio Calyequita
1240  Xochimilco     Cerrillos Primera Seccion
1241  Xochimilco     Cerrillos Segunda Seccion
1242  Xochimilco     Cerrillos Tercera Seccion
1243  Xochimilco                    Cristo Rey
1244  Xochimilco                    Del Carmen
1257  Xochimilco                    Las Animas
1258  Xochimilco                   Las Mesitas
1260  Xochimilco            Lomas De Nativitas
1262  Xochimilco                     Nativitas
1277  Xochimilco  Pueblo Santa Maria Nativitas
1283  Xochimilco               Quirino Mendoza
1288  Xochimilco                    San Felipe
1289  Xochimilco                    San Isidro
1298  Xochimilco                   Santiaguito

Exportado a 'tulyehualco_aoi_oficial.geojson' con éxito.
